**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/gillespie-2024"], check=True)
    os.chdir("/content/encore/entries/gillespie-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Gillespie et al. (2024) on DANDI:000115

Paper: Gillespie, Astudillo Maya, Denovellis, Desse and Frank (2024), "Neurofeedback training can modulate task-relevant memory replay rate in rats", eLife 12, RP90944, doi:10.7554/eLife.90944. The claims file (`claims/gillespie-2024.yaml`) was written from the version of record (PMC11221834) with its figure legends and supplements.

Datasets, streamed from the DANDI Archive (no download):

- DANDI:000115, version 0.210914.1732 (doi:10.48324/dandi.000115/0.210914.1732), the control cohort (rats jaq, roquefort, despereaux, montague), reused from Gillespie et al. 2021. The paper cites the dandiset without a version; 0.210914.1732 is its only published version, and the draft has the same 57 assets.
- DANDI:000629, version 0.240627.0646 (doi:10.48324/dandi.000629/0.240627.0646), the neurofeedback cohort (remy, gus, bernard, fievel), deposited for this paper. It is not listed for the paper in the registry, so it is used here as an input, as docs/protocol.md (Author, step 1.4) says for a dataset the paper's authors deposited for the paper; the claims that need it are `data: elsewhere` in the claims file and are counted apart from those on 000115; it is its only published version. Its behavioural claims are tested on rats 2 to 4; rat 1's port events cannot be used (cell `finding-nf-dio-corrupt`).

Both dandisets hold, per session, the raw 30 kHz broadband signal of 128 channels, head LED position, port digital events (DIO) and statescript logs. They hold no LFP, SWR detections, sorted spikes or decodes. The electrode table's location is `None` for every channel; the electrode groups (tetrodes) carry a location (in the file of cell `finding-no-lfp-no-ca1-labels`, `ca1` for 23 of 32, `bad_ca1`, empty, `can1ref` or `lost` for the others), which does not say which tetrodes are in the CA1 cell layer. Every claim that needs SWRs or replay would need SWR detection (and, for replay, clusterless decoding) run on the raw data: about 9 TB for 000115 and 17 TB for 000629. These claims are blocked as `too_large`, with the estimate in the cell tagged `too-large-estimate`, which also times the detection's input on a short excerpt of the raw signal. The claims on the control cohort's behaviour are tested from the DIO events and position.

Claims on DANDI:000115 (9 claims: 5 tested, 4 blocked):

| Claim | Tested here |
|---|---|
| control-cohort-trial-counts | tested |
| control-decode-counts | tested |
| fig2s1h-control-dwell-increases | tested |
| fig3a-control-speed | tested |
| fig5s1-control-search-efficiency-increases | tested |
| fig2s1c-control-prereward-swr-counts, fig2s1f-control-rate-increase, fig2b-control-timecourse | blocked, too_large (SWR detection on raw data) |
| fig4g-control-replay-glm | blocked, too_large (clusterless decoding on raw data) |

Claims whose data are elsewhere, on the neurofeedback cohort in DANDI:000629 (26 claims: 7 tested, 19 blocked):

| Claim | Tested here |
|---|---|
| fig1s1c-prereward-dwell-matched, fig2e-postreward-dwell, fig3a-speed-below-immobility, fig5a-search-efficiency, fig5b-redundant-search-balanced, fig5c-repeat-accuracy-unchanged, fig5d-repeat-errors-balanced | tested on three of the four neurofeedback rats (gus, bernard, fievel); the port events of rat1 (remy) are corrupted |
| The 19 claims on the neurofeedback cohort's SWRs, replay and timecourses, fig4s1-replay-glm-pre-post among them | blocked, too_large |


In [ ]:
import sys, time, resource
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import nfbehavior as nb
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
RATS = nb.CONTROL_RATS
t0 = time.time()
trials, info = nb.load_trials(nb.CONTROL)
print(f"loaded {len(trials)} center-port trials from {len(info)} run epochs in {time.time()-t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## Data check

What was loaded, counted by rat, session and epoch, with the ranges and units of every variable used. Run epochs are the epochs whose tag contains `_r`; every run epoch of 000115 is the task `Eight arm flexible spatial task`, described `goal_delay`. The files carry no flag for forage assist, the online SWR threshold (ripthresh) or decode quality (gooddecode), the flags the paper's code uses to select epochs.

In [ ]:
A = nb.assets(nb.CONTROL)
print("assets per rat:", A.groupby("subject").size().to_dict())
print("run epochs per rat:", info.groupby("rat").size().to_dict())
print("sessions (days) per rat:", info.groupby("rat").day.nunique().to_dict())
print("run epoch duration (min) by rat:")
print(info.groupby("rat").duration_min.describe().round(1))
print("center-port trials (with a pre-reward period) per rat:", trials.groupby("rat").size().to_dict())
print("of which lockout after center success:", trials.groupby("rat").lockout.sum().to_dict())
print("trials per center port:", trials.groupby(["rat", "port"]).size().unstack().to_dict())
print()
print("position conversion factors (m per unit) of the run series:", info.pos_conv.value_counts().to_dict())
print("x range of head position in run epochs (cm): %.1f to %.1f" % (info.pos_range_cm.min(), info.pos_range_cm.max()))
for c in ["pre", "post_a", "post_b"] + [f"vel{s}" for s in nb.SPEED_SIGMAS]:
    x = trials[c]
    print(f"{c:8s} (s or cm/s): min {x.min():.2f}, median {x.median():.2f}, max {x.max():.2f}, NaN {x.isna().sum()}")
print("trials with a pre-reward period <= 0:", int((trials.pre <= 0).sum()))
ph = trials.phase.dropna()
print("task phase labels: search new", int((ph == 0).sum()), "search redundant", int((ph == -0.2).sum()),
      "rewarded", int((ph % 1 == 0).sum() - (ph == 0).sum()), "repeat error", int(((ph % 1 > 0.2) & (ph % 1 < 0.4)).sum()),
      "old-goal closer", int(((ph % 1 > 0.85)).sum()))
print()
print("pre-reward window, from the first entry at the port to the pump: trials with more than one entry, and with a withdrawal over 200 ms")
w = trials.groupby("rat").agg(trials=("entries", "size"), multi_entry=("entries", lambda x: int((x > 1).sum())),
                              out_over_200ms=("out_max", lambda x: int((x > nb.PROX_TIME).sum())),
                              median_pre_first=("pre", "median"), median_pre_last=("pre_last", "median"))
print(w.round(2).to_string())


Clock check. The DIO events and the position share the file's clock if the head is at the rewarded center port when that port's pump turns on. For each epoch the two ports' locations are the median head positions at their reward times; the port of each trial is then told from the head position at the reward time plus a shift. If the streams were offset, the accuracy would peak away from zero.

In [ ]:
acc = {s: nb.port_decoding(trials, s) for s in nb.CLOCK_SHIFTS}
print("port told from head position at reward + shift (s): accuracy")
for s, a in acc.items():
    print(f"  {s:+4d} s: {a:.3f}")
per_rat = {r: nb.port_decoding(trials[trials.rat == r], 0) for r in RATS}
per_rat_m10 = {r: nb.port_decoding(trials[trials.rat == r], -10) for r in RATS}
print("per rat at 0 s:", {k: round(v, 3) for k, v in per_rat.items()}, " at -10 s:", {k: round(v, 3) for k, v in per_rat_m10.items()})
worst = [nb.port_decoding(d, 0) for _, d in trials.groupby(["rat", "day", "epoch"])]
print("epochs with accuracy below 0.9 at 0 s:", int(np.sum(np.array(worst) < 0.9)), "of", len(worst), "; lowest", round(min(worst), 3))

Duplicate check: an epoch deposited twice would repeat its home-reward intervals. Each run epoch's first five intervals between home rewards are compared across all epochs.

In [ ]:
dup = info[info.duplicated("fingerprint", keep=False)]
print("run epochs whose first five home-reward intervals repeat another epoch's:", len(dup))
print(dup[["rat", "day", "epoch", "fingerprint"]] if len(dup) else "none")
st = info.groupby("rat").apply(lambda d: d.start.is_monotonic_increasing, include_groups=False)
print("epoch start times increase with day number for every rat:", st.to_dict())

### What the files hold for the SWR analyses

The paper detects SWRs on the LFP of all CA1 cell layer tetrodes. The cell below lists what one file holds: the acquisition has only the raw ElectricalSeries, there is no LFP or SWR table under processing, and the electrode table's `location` is `None` for every channel. The electrode groups, one per tetrode, carry a location; in the file below, `ca1` for 23 of 32, `bad_ca1` for 4, empty for 3, `can1ref` for 1 and `lost` for 1. Whether a `ca1` tetrode is in the cell layer is not said.

In [ ]:
r = A.iloc[0]
f = nb.open_nwb(r.asset_id)
print(r.path)
print("acquisition:", list(f["acquisition"].keys()), "processing:", list(f["processing"].keys()), "analysis:", list(f["analysis"].keys()))
e = f["acquisition/e-series/data"]
print("e-series:", e.shape, e.dtype, "chunks", e.chunks)
el = f["general/extracellular_ephys/electrodes"]
print("electrode location values:", {v.decode() for v in el["location"][:]})
print("electrode columns:", list(el.keys()))
eg = f["general/extracellular_ephys"]
locs = pd.Series({k: eg[k].attrs.get("location") for k in eg if k != "electrodes"})
print("electrode group locations:", locs.value_counts().to_dict())

In [ ]:
print("run epochs with NaN position timestamps:", int((info.pos_nan_ts > 0).sum()), "of", len(info),
      "; NaN timestamps per such epoch:", sorted(info.pos_nan_ts[info.pos_nan_ts > 0].unique().tolist()))

### The neurofeedback cohort (DANDI:000629)

The claims on the neurofeedback cohort's SWRs and replay are blocked (the summary says why); its behavioural claims are tested below on rats 2 to 4. The checks below read its port DIO and metadata, as a user of the dandiset would. Port pumps fire once per reward, so a run epoch has some tens to a few hundred onsets per pump and a sleep epoch has none.

In [ ]:
t0 = time.time()
nfdio = nb.survey_dio(nb.NF)
print(f"read {nfdio[['rat','day']].drop_duplicates().shape[0]} sessions in {time.time()-t0:.0f} s")
bad = nfdio[nfdio.homepump > 1000]
print("epochs with more than 1000 home pump onsets, by rat and epoch kind:")
print(bad.groupby(["rat", "sleep"]).size())
print("sessions with such an epoch:", bad.groupby("rat").day.apply(lambda d: sorted(set(d))).to_dict())
print()
print("sleep epochs with pump onsets, by rat:", nfdio[nfdio.sleep & (nfdio.homepump + nfdio.Rpump + nfdio.Wpump > 0)].groupby("rat").size().to_dict())
x = nfdio[(nfdio.rat == "remy") & (nfdio.day == 33)][["epoch", "minutes", "homepump", "Rpump", "Wpump", "t22"]]
print("remy, day 33:"); print(x.round(1).to_string(index=False))
y = nfdio[(nfdio.rat == "bernard") & (nfdio.day == 25)][["epoch", "minutes", "homepump", "Rpump", "Wpump", "t22"]]
print("bernard, day 25, for comparison:"); print(y.round(1).to_string(index=False))
NAall = nb.assets(nb.NF)
for rat, day in (("remy", 33), ("bernard", 25)):
    g = nb.open_nwb(NAall[(NAall.subject == rat) & (NAall.day == day)].asset_id.iloc[0])
    hp = g["processing/behavior/behavioral_events/homepump"]
    d, t = hp["data"][:], hp["timestamps"][:]
    r1 = nb.run_epochs(g).iloc[0]
    k = np.where((t >= r1.start) & (t <= r1.stop))[0][:12]
    print(f"{rat} day {day}, first homepump DIO events of {r1.tag} (time from epoch start, s; value):",
          [(round(float(t[i] - r1.start), 3), int(d[i])) for i in k])

In [ ]:
print("position conversion factors in 000629:", sorted({c for v in nfdio.pos_conversion for c in v}))
z = nfdio[nfdio.session_start.str.startswith("1970")]
print("sessions of 000629 whose session_start_time is 1970-01-01:", z.groupby("rat").day.apply(lambda d: sorted(set(d))).to_dict(),
      "of", nfdio[["rat", "day"]].drop_duplicates().shape[0])
print("task descriptions in 000629:", sorted({c for v in nfdio.task_descriptions for c in v}))
print("sessions with trigger (t22) onsets in a run epoch, by rat:",
      nfdio[~nfdio.sleep & (nfdio.t22 > 0)].groupby("rat").day.nunique().to_dict(),
      "of", nfdio.groupby("rat").day.nunique().to_dict())

### Neurofeedback cohort trials

The trials of rats 2 to 4 (gus, bernard, fievel) are parsed as for the control cohort, from their last 15 sessions, run epochs of the task `goal` only (the code's `environment goal`; the other run epochs are named `hybrid`, `hybrid_shifted` or `bad`). The files do not mark the neurofeedback phase, the detection threshold or the plateau (stable) period that the paper analyses ("All quantification of manipulation cohort trials ... use only trials from the stable period of neurofeedback training", Methods, Statistical tests). The plateau is read three ways: `matched`, the final whole days whose trial counts at the two center ports are closest to the Figure 2C legend counts (684/640, 1157/1201, 1602/1552 NF/delay trials for rats 2 to 4); `last7` and `last10`, the final 7 or 10 days (Methods: the threshold "is maintained at the maximum threshold for 7-10 days"). The position series of 000629 have a conversion factor of 0; the 0.2 cm per pixel of 000115's run camera is used, as the maze is the same (Methods, Behavioral training, "the same as used in Gillespie et al., 2021"). This is an assumption: with it, the x range of the head positions per run epoch has a smaller median in 000629 than in 000115 but a wider spread and a larger maximum (the next cell prints both distributions), so the scale may differ. The comparisons of NF with delay trials do not depend on it; the 4 cm/s cutoff and the comparisons with control trials do.

In [ ]:
NFRATS = ["gus", "bernard", "fievel"]  # rat2 to rat4 of the paper
PAPER_COUNTS = {"remy": (1892, 2022), "gus": (684, 640), "bernard": (1157, 1201), "fievel": (1602, 1552)}  # Figure 2C legend, NF and delay
READINGS = ["matched", "last7", "last10"]
NA = nb.assets(nb.NF)
NA = NA[NA.subject.isin(NFRATS)].sort_values("day").groupby("subject").tail(15)
t0 = time.time()
nft, nfinfo = nb.load_trials(nb.NF, a=NA, px_cm=0.2, env="goal")
print(f"loaded {len(nft)} trials from {len(nfinfo)} goal run epochs of {NA.shape[0]} sessions in {time.time()-t0:.0f} s")
plat = {r: nb.plateau_days(nft, r, READINGS, PAPER_COUNTS[r]) for r in NFRATS}
rows = []
for r in NFRATS:
    for rd in READINGS:
        d = nft[(nft.rat == r) & nft.day.isin(plat[r][rd])]
        rows.append(dict(rat=r, reading=rd, days=f"{min(plat[r][rd])} to {max(plat[r][rd])}", ndays=len(plat[r][rd]),
                         epochs=d[["day", "epoch"]].drop_duplicates().shape[0],
                         nf=int((d.port == "R").sum()), delay=int((d.port == "W").sum()),
                         paper_nf=PAPER_COUNTS[r][0], paper_delay=PAPER_COUNTS[r][1]))
print(pd.DataFrame(rows).to_string(index=False))
def sel(rd):
    return pd.concat([nft[(nft.rat == r) & nft.day.isin(plat[r][rd])] for r in NFRATS])
m = sel("matched")
print("plateau (matched) ranges: pre %.2f to %.2f s, post (b) %.2f to %.2f s, speed (0.25 s) %.2f to %.2f cm/s, head x range %.1f to %.1f cm" % (
      m.pre.min(), m.pre.max(), m.post_b.min(), m.post_b.max(), m["vel0.25"].min(), m["vel0.25"].max(),
      nfinfo.pos_range_cm.min(), nfinfo.pos_range_cm.max()))
print("x range of head position per run epoch (cm), quantiles 0, 0.1, 0.5, 0.9, 1: 000115",
      info.pos_range_cm.quantile([0, .1, .5, .9, 1]).round(1).tolist(), "; 000629 at 0.2 cm/px",
      nfinfo.pos_range_cm.quantile([0, .1, .5, .9, 1]).round(1).tolist())
print("pre-reward withdrawals in the plateau (matched): trials with more than one entry", m.groupby("rat").entries.apply(lambda x: int((x > 1).sum())).to_dict(),
      "; with a withdrawal over 200 ms", m.groupby("rat").out_max.apply(lambda x: int((x > nb.PROX_TIME).sum())).to_dict(), "; of", m.groupby("rat").size().to_dict())
print("lockouts after center success in the plateau:", m.groupby("rat").lockout.sum().to_dict())
print("port decoding from head position at reward (clock check), 0 s:", round(nb.port_decoding(m, 0), 3), "; -10 s:", round(nb.port_decoding(m, -10), 3))

Which center port is the neurofeedback port. The statescript logs count rewards as `ripCount` and `waitCount`, and the DIO names the ports R and W. The Methods say delay times "are chosen from the latencies until trigger SWR generation from the 8 prior neurofeedback trials", so each delay-port wait should match one of the 8 previous waits at the neurofeedback port. The cell tests this both ways.

In [ ]:
y = nb.yoking(sel("matched"))
for port, other in (("W", "R"), ("R", "W")):
    v = y[port]
    print(f"{port} trials: n {len(v)}, median smallest |difference| to the 8 previous {other} waits {np.median(v):.3f} s, "
          f"fraction within 0.1 s {np.mean(v < 0.1):.3f}")
print("so R is the neurofeedback port (type 1) and W the delay port (type 2)")

### Size of the SWR and replay analyses

The SWR detection the paper describes needs the ripple-band LFP of every CA1 cell layer tetrode over each whole behavioural epoch (the threshold is 2 sd above the epoch mean). The raw series is stored in chunks of one channel by about 9 s, so one channel per tetrode can be read alone. The cell times the reading of one channel and scales it to the run epochs. It also times the input of the detection on a subset: 180 s of one channel of each of 10 `ca1` tetrodes, fetched by parallel HTTP range requests (h5py over the stream reads one chunk at a time, whatever the number of threads) and decimated to 1500 Hz, and scales it to two run epochs per control rat, the smallest subset that gives the Figure 2B bins their 100 trials, and to one run epoch per control rat, the least a per-rat sign needs. The notebook limit is 60 min (docs/protocol.md, Author, step 3, "Keep it fast").

In [ ]:
rate, shape, chunks, dtype = nb.raw_read_rate(A.iloc[1].asset_id)
run_s = info.duration_min.sum() * 60
gb = run_s * 30000 * 2 * 30 / 1e9  # one channel per tetrode, 30 tetrodes, int16
print(f"one-channel read rate: {rate:.1f} MB/s (series {shape}, chunks {chunks}, {dtype})")
print(f"control run epochs: {run_s/3600:.1f} h; one channel of each of 30 tetrodes: {gb:.0f} GB")
for streams in (1, 16, 64):
    print(f"  at {streams} parallel streams: {gb*1e3/(rate*streams)/3600:.1f} h")
nf_run_h = nfdio[~nfdio.sleep].minutes.sum() / 60
print(f"neurofeedback cohort run epochs: {nf_run_h:.1f} h, {nf_run_h*3600*30000*2*30/1e9:.0f} GB on the same basis")
print("clusterless decoding needs all 4 channels of every tetrode at 30 kHz: the whole files, "
      f"{A['size'].sum()/1e12:.1f} TB for 000115 and {nb.assets(nb.NF)['size'].sum()/1e12:.1f} TB for 000629")
from concurrent.futures import ThreadPoolExecutor
def h5_read(ch, aid=A.iloc[5].asset_id):
    e = nb.open_nwb(aid)["acquisition/e-series/data"]
    return e[e.chunks[0] * 50: e.chunks[0] * 53, ch].nbytes
for w in (4, 16):
    t1 = time.time()
    with ThreadPoolExecutor(w) as ex:
        mb = sum(ex.map(h5_read, range(0, 4 * w, 4))) / 1e6
    print(f"h5py reads of {w} channels in {w} threads: {mb / (time.time() - t1):.1f} MB/s together")
r5 = A.iloc[5]
g = nb.open_nwb(r5.asset_id)
chans, nca1 = nb.ca1_channels(g, 10)
e5 = nb.run_epochs(g).iloc[0]
t1 = time.time()
tt, X = nb.ripple_lfp(r5.asset_id, e5.start + 600, e5.start + 780, chans)
dt = time.time() - t1
print(f"{r5.path}: {nca1} ca1 tetrodes; 180 s of 10 of them read and decimated in {dt:.0f} s ({X.shape} at {nb.LFP_FS} Hz)")
sub_s = info.groupby("rat").duration_min.median().sum() * 2 * 60
print(f"two run epochs per control rat: {sub_s/3600:.1f} h, about {sub_s / 180 * dt / 3600:.1f} h at this rate in one process, "
      f"{sub_s / 180 * dt / 3600 / 4:.1f} h in four")
print(f"one run epoch per control rat: {sub_s/2/3600:.1f} h, about {sub_s / 2 / 180 * dt / 3600 / 4:.1f} h in four processes; "
      f"the notebook limit is 60 min")

## control-cohort-trial-counts

> The control cohort data used for the center port analyses comprise 2490, 2629, 2027 and 3021 trials for the four control rats.

Trials with a pre-reward period: a center port reward reached from a home port reward, with no lockout, or followed by a lockout (dfa_ripquantpertrial_allphase_NF.m, `rw` phase). The files carry no ripthresh, forageassist or environment flags, so reading (a) of the criterion is used: all run epochs of the full delay task (all are `goal_delay`). Reading (b) removes epochs marked forage assist or no delay; none is marked, so (b) is the same set.

In [ ]:
exp = {"jaq": 2490, "roquefort": 2629, "despereaux": 2027, "montague": 3021}
n = trials.groupby("rat").size()
tab = pd.DataFrame({"measured": n.reindex(RATS), "paper": pd.Series(exp)})
tab["tolerance"] = (tab.paper * 0.1).round()
tab["within"] = (tab.measured - tab.paper).abs() <= tab.tolerance
tab["epochs"] = info.groupby("rat").size().reindex(RATS)
print(tab)
print("within tolerance:", int(tab.within.sum()), "of 4")
print("without trials whose pre-reward period is under 1 s:", trials[trials.pre >= 1].groupby("rat").size().reindex(RATS).to_dict())

**Result: reproduced.** All run epochs give 2557, 2663, 2042 and 3081 trials with a pre-reward period for jaq, roquefort, despereaux and montague, within 3% of the paper's 2490, 2629, 2027 and 3021 and inside the 10% tolerance for all four rats. The files mark no forage-assist epochs, so readings (a) and (b) are the same set. The trials come from a trial parser written here (helper docstring), an assumption.

## control-decode-counts

> The control cohort data used for the replay and behavior analyses comprise 24, 32, 23 and 33 behavioral epochs, 2058, 2509, 1879 and 2795 trials for remote replay rate, and 1458, 1636, 1464 and 2181 trials for the replay GLM.

Figure 4E trials are trials with a task phase (non-lockout trials, dfa_ripcontent_nf.m); Figure 4G trials are those with a task phase from the second goal block of the epoch on (Methods, Replay content analysis). The dandiset does not mark gooddecode, so the epochs are those of control-cohort-trial-counts, and by the criterion the epoch counts are then reported as not comparable.

In [ ]:
nl = trials[~trials.lockout & trials.phase.notna()]
e4 = {"jaq": 24, "roquefort": 32, "despereaux": 23, "montague": 33}
t4e = {"jaq": 2058, "roquefort": 2509, "despereaux": 1879, "montague": 2795}
t4g = {"jaq": 1458, "roquefort": 1636, "despereaux": 1464, "montague": 2181}
tab = pd.DataFrame({"epochs": info.groupby("rat").size(), "epochs_paper": pd.Series(e4),
                    "trials_4E": nl.groupby("rat").size(), "paper_4E": pd.Series(t4e),
                    "trials_4G": nl[nl.block >= 1].groupby("rat").size(), "paper_4G": pd.Series(t4g)}).reindex(RATS)
tab["epochs_ok"] = (tab.epochs - tab.epochs_paper).abs() <= 0.1 * tab.epochs_paper
tab["4E_ok"] = (tab.trials_4E - tab.paper_4E).abs() <= (0.1 * tab.paper_4E).round()
tab["4G_ok"] = (tab.trials_4G - tab.paper_4G).abs() <= (0.1 * tab.paper_4G).round()
print(tab)
print("4E within tolerance:", int(tab["4E_ok"].sum()), "of 4; 4G:", int(tab["4G_ok"].sum()), "of 4")
print("relative difference 4E:", ((tab.trials_4E / tab.paper_4E - 1) * 100).round(1).to_dict())
print("relative difference 4G:", ((tab.trials_4G / tab.paper_4G - 1) * 100).round(1).to_dict())
# jaq has three more epochs than the paper's decode set: the 4E count per epoch, for scale
per_ep = nl.groupby(["rat", "order"]).size()
print("jaq task-phase trials per epoch, median:", per_ep.loc["jaq"].median(),
      "; jaq count without its 3 smallest epochs:", int(per_ep.loc["jaq"].sum() - per_ep.loc["jaq"].nsmallest(3).sum()),
      "; without its first 3 epochs:", int(per_ep.loc["jaq"].iloc[3:].sum()))

**Result: partially reproduced.** The dandiset holds 27, 32, 24 and 33 run epochs where the paper's decode analyses use 24, 32, 23 and 33; as gooddecode is not marked, the epoch counts are not comparable under the criterion. Trials with a task phase are 2366, 2611, 2018 and 2974 (paper 2058, 2509, 1879 and 2795) and from the second goal block on 1617, 1601, 1530 and 2246 (paper 1458, 1636, 1464 and 2181): both sets match for roquefort, despereaux and montague and miss for jaq (+15.0% and +10.9%), the rat with three more epochs than the paper's decode set. Without jaq's first three epochs its Figure 4E count is 2076.

## fig2s1h-control-dwell-increases

> In the control cohort, the time spent at the center port after reward delivery tended to increase as subjects gained experience with the task.

Per control rat, each epoch's mean post-reward duration against the epoch's order, Pearson correlation (dfs_rips_vsctrlrats.m part 10). Leaving the port is read two ways (helper docstring): (a) the first beam-off lasting more than 200 ms, (b) the last beam-off before the next port visit.

In [ ]:
rows = []
for col in ("post_a", "post_b"):
    for r in RATS:
        d = trials[trials.rat == r].groupby("order")[col].mean()
        rr, p = stats.pearsonr(d.index, d.values)
        rows.append(dict(reading=col, rat=r, epochs=len(d), r=round(rr, 3), r2=round(rr**2, 3), p=p,
                         first5_mean=round(d.iloc[:5].mean(), 2), last5_mean=round(d.iloc[-5:].mean(), 2),
                         range=f"{d.min():.1f} to {d.max():.1f}"))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("positive r:", res.groupby("reading").apply(lambda d: int((d.r > 0).sum()), include_groups=False).to_dict())
print("trials without a leaving time:", int(trials.post_a.isna().sum()), "(a),", int(trials.post_b.isna().sum()), "(b)")
fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for i, r in enumerate(RATS):
    d = trials[trials.rat == r].groupby("order").post_a.mean()
    ax[i].plot(d.index, d.values, ".-"); ax[i].set_title(r); ax[i].set_xlabel("epoch"); ax[i].set_ylabel("post-reward (s)")
plt.tight_layout()

**Result: partially reproduced.** With leaving the port read as the first exit longer than 200 ms (reading a), roquefort's r is -0.129 (p = 0.48) and the other three rats are positive (0.605, 0.669 and 0.511), which is partially reproduced. With leaving read as the last beam-off before the next port visit (reading b), r is positive in all four rats (0.495, 0.319, 0.648 and 0.516; p = 0.0087, 0.076, 0.0006 and 0.0021), which would be reproduced. Neither the paper nor its code defines leaving the port, and the files support both readings (the statescript's 200 ms allowance for reading a, the next port visit for reading b), so the outcome is the one both readings give: r is positive in at least three of the four rats.

## fig3a-control-speed

> In the control cohort, mean head speed during the pre-reward nosepoke was below the 4 cm/s immobility cutoff.

Per-trial mean of the smoothed head speed over the pre-reward period (dfa_ripquantpertrial_allphase_NF.m, meanvel). Readings of the criterion: (a) the median over trials of the per-trial mean, (b) the mean over trials; and (i) a speed the dandiset provides, (ii) speed computed from the head LEDs. The dandiset provides no speed, so (i) cannot be run; (ii) is run with Gaussian smoothing of sd 0.1, 0.25 and 0.5 s (helper docstring).

In [ ]:
rows = []
for s in nb.SPEED_SIGMAS:
    for r in RATS:
        v = trials.loc[trials.rat == r, f"vel{s}"]
        rows.append(dict(sigma_s=s, rat=r, n=int(v.notna().sum()), median=round(v.median(), 2), mean=round(v.mean(), 2),
                         p90=round(v.quantile(.9), 2), frac_above_4=round((v > 4).mean(), 3)))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("rats below 4 cm/s, median reading:", res.groupby("sigma_s").apply(lambda d: int((d["median"] < 4).sum()), include_groups=False).to_dict())
print("rats below 4 cm/s, mean reading:", res.groupby("sigma_s").apply(lambda d: int((d["mean"] < 4).sum()), include_groups=False).to_dict())
print("trials without position in the window:", int(trials["vel0.25"].isna().sum()))
print("pre-reward period from the entry after the last withdrawal over 200 ms (other reading of RWstart): largest median and mean over rats, by smoothing:",
      {s: (round(trials.groupby("rat")[f"vel{s}_last"].median().max(), 2), round(trials.groupby("rat")[f"vel{s}_last"].mean().max(), 2)) for s in nb.SPEED_SIGMAS})

**Result: reproduced.** Under every smoothing (Gaussian sd 0.1, 0.25 and 0.5 s) and both summaries, the mean head speed over the pre-reward period is below 4 cm/s in all four control rats: medians 2.27 to 3.02 cm/s at 0.1 s and 0.88 to 1.36 cm/s at 0.5 s, means 2.32 to 3.01 and 1.05 to 1.59. The dandiset provides no speed of its own, so reading (i) of the criterion could not be run. With the pre-reward period started at the entry after the last withdrawal over 200 ms (the other reading of RWstart), the largest median and mean over rats are 3.02 and 3.01 cm/s at 0.1 s smoothing, the same as above, since only 3 control trials have such a withdrawal (sanity cell).

## fig5s1-control-search-efficiency-increases

> In the control cohort, search efficiency tends to increase over behavioral epochs.

Per epoch, the fraction of search trials (task phase <= 1) that are not errors (fractional part of the phase above 0 and below 0.85), against the epoch's order, Pearson correlation (dfs_ripcontent_vsctrlrats.m part 5). The gooddecode epochs are not marked, so all run epochs are used (the criterion's variant). A redundant search trial is read two ways (helper docstring): (a) a revisit of an arm already searched in the block, (b) also counting the arm of the trial that closed the previous block.

In [ ]:
rows = []
for col in ("phase", "phase_b"):
    for r in RATS:
        d = trials[(trials.rat == r) & trials[col].notna()]
        g = d.groupby("order")[col]
        def eff(x):
            s = x[x <= 1]
            if len(s) == 0:
                return np.nan
            frac = np.mod(s, 1)
            return float((~((frac > 0) & (frac < 0.85))).mean())
        e = g.apply(eff).dropna()
        rr, p = stats.pearsonr(e.index, e.values)
        rows.append(dict(reading=col, rat=r, epochs=len(e), r=round(rr, 3), r2=round(rr**2, 3), p=p,
                         first5=round(e.iloc[:5].mean(), 3), last5=round(e.iloc[-5:].mean(), 3),
                         search_trials=int((d[col] <= 1).sum())))
res = pd.DataFrame(rows)
print(res.to_string(index=False))
print("positive r:", res.groupby("reading").apply(lambda d: int((d.r > 0).sum()), include_groups=False).to_dict())
# goal carried over from the previous epoch of the same day (the first goal of an epoch is found by search)
co = []
for (r, day), g in trials[~trials.lockout & trials.armrew].sort_values("rwend").groupby(["rat", "day"]):
    eps = list(dict.fromkeys(g.epoch))
    for a, b in zip(eps[:-1], eps[1:]):
        last, first = g[g.epoch == a].arm.iloc[-1], g[g.epoch == b].arm.iloc[0]
        nsearch = int(((trials.rat == r) & (trials.day == day) & (trials.epoch == b) & (trials.block <= 0) & (trials.phase <= 1)).sum())
        co.append(dict(rat=r, day=day, epoch=b, same_goal=last == first, first_block_search=nsearch))
co = pd.DataFrame(co)
print("epochs that follow another run epoch of the same day:", co.groupby("rat").size().to_dict(),
      "; of which the first goal is the previous epoch's last goal:", co.groupby("rat").same_goal.sum().astype(int).to_dict())
print("search trials before the first reward of those epochs:", co[co.same_goal].groupby("rat").first_block_search.sum().to_dict(),
      "of", res[res.reading == "phase"].set_index("rat").search_trials.to_dict(), "search trials")
drop = set(map(tuple, co[co.same_goal][["rat", "day", "epoch"]].values))
rows = []
for col in ("phase", "phase_b"):
    for r in RATS:
        d = trials[(trials.rat == r) & trials[col].notna()]
        d = d[[(r, dy, ep) not in drop for dy, ep in zip(d.day, d.epoch)]]
        e = d.groupby("order")[col].apply(eff).dropna()
        rr, p = stats.pearsonr(e.index, e.values)
        rows.append(dict(reading=col, rat=r, epochs=len(e), r=round(rr, 3), p=p))
print("without the epochs whose first goal was carried over:")
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced, resting on an assumption.** The per-epoch fraction of non-redundant search trials correlates positively with epoch order in all four rats under both readings of a redundant search (r = 0.143, 0.333, 0.266, 0.301 under (a); 0.156, 0.295, 0.224, 0.334 under (b)). No single correlation reaches p < 0.05 (smallest p = 0.058); the paper reports no statistic and calls the increase subtle. All run epochs are used, as gooddecode is not marked, and the task phases come from rules written here. The first goal of an epoch is the previous epoch's last goal in 8 of the 59 epochs that follow another run epoch of the same day; these hold 39 of the 3201 search trials (2, 6, 28 and 3 per rat). Without these epochs r stays positive in all four rats under both readings (0.116, 0.357, 0.301, 0.302 under (a); 0.119, 0.338, 0.248, 0.323 under (b)).

## fig1s1c-prereward-dwell-matched

> Time spent at the center ports before reward was approximately matched between NF and delay trials by the yoked delay design.

Per-trial pre-reward duration (dfs_rips_vsctrlrats.m part 6); criterion: per-rat median NF and delay durations differ by less than 10%, with the ranksums and mixed models reported. Rats 2 to 4 only; each plateau reading.

In [ ]:
ctrl = {r: trials[trials.rat == r] for r in RATS}
def split(m, col):
    return ({r: m[(m.rat == r) & (m.port == "R")][col].values for r in NFRATS},
            {r: m[(m.rat == r) & (m.port == "W")][col].values for r in NFRATS})
for rd, col in [(rd, "pre") for rd in READINGS] + [("matched", "pre_last")]:
    nf, dl = split(sel(rd), col)
    t = nb.ranksums_bh(nf, dl)
    t["diff_pct"] = (t.median_nf / t.median_delay - 1) * 100
    c = {r: ctrl[r][col].values for r in RATS}
    l1, l2 = nb.lme_cohort(c, nf), nb.lme_cohort(c, dl)
    print(f"plateau {rd}, {col}:"); print(t.round(4).to_string(index=False))
    print(f"  mixed model NF vs control: coefficient {l1[0]:.2f} s, p {l1[1]:.3g}; delay vs control: {l2[0]:.2f} s, p {l2[1]:.3g}")
    print(f"  rats with |difference| < 10%: {int((t.diff_pct.abs() < 10).sum())} of 3")

**Result: not reproduced, resting on an assumption, on three of the four rats.** Under the adopted plateau reading (matched), the median NF and delay pre-reward durations differ by -1.1% (gus), 10.2% (bernard) and 7.9% (fievel), so two of three rats are inside the criterion's 10%. Bernard and fievel differ significantly (corrected p = 0.0035 and 0.0014), as the paper's own tests found for rats 3 and 4. Under last7 all three rats are within 10% (4.8%, 8.4%, 4.1%), and under last10 one is (8.2%, 10.2%, 11.1%). With the pre-reward period started at the entry after the last withdrawal over 200 ms (110 of bernard's 2366 plateau trials and 14 of fievel's 3166 have one, none of gus's), bernard's difference is 16.1% and the other two are unchanged (-1.1%, 7.8%), again two of three. Both trial types are longer than control trials (mixed models: +3.83 s, p = 2.0e-08; +3.12 s, p = 6.35e-05). Rat 1 (remy) cannot be tested.

The criterion asks for all four rats within 10% and names no partial level, the rule by which fig5b-redundant-search-balanced is judged. Under the adopted reading bernard is outside 10% (10.2%, and 16.1% with the other pre-reward start), so the claim is not reproduced. The outcome rests on the plateau reading: under last7 all three rats are within 10%. Bernard's 10.2% is just over a threshold the claims file chose, while the paper finds significant differences in rats 3 and 4 and still calls the durations approximately matched; this is asked of a reader in questions/gillespie-2024.yaml.

## fig2e-postreward-dwell

> Post-reward dwell time was slightly but significantly shorter on NF than on delay trials, and manipulation subjects stayed at the port about half as long as control subjects.

Per-trial post-reward duration (dfs_rips_vsctrlrats.m part 3); ranksums per rat with Benjamini-Hochberg, mixed models against the control cohort; the ratio is that of the inset means (mean over rats of each rat's median), with the ratio of means of per-trial durations as a variant. Each plateau reading and both readings of leaving the port.

In [ ]:
def inset(d):
    return np.mean([np.nanmedian(v) for v in d.values()])
rows = []
for col in ("post_b", "post_a"):
    for rd in READINGS:
        nf, dl = split(sel(rd), col)
        c = {r: ctrl[r][col].values for r in RATS}
        t = nb.ranksums_bh(nf, dl)
        l1, l2 = nb.lme_cohort(c, nf), nb.lme_cohort(c, dl)
        ci, ni, di = inset(c), inset(nf), inset(dl)
        cm = np.nanmean(np.concatenate(list(c.values())))
        rows.append(dict(leave=col, plateau=rd, nf_shorter_all=bool((t.median_nf < t.median_delay).all()),
                         max_p_bh=t.p_bh.max(), lme_nf_coef=l1[0], lme_nf_p=l1[1], lme_delay_coef=l2[0], lme_delay_p=l2[1],
                         inset_ctrl=ci, inset_nf=ni, inset_delay=di, ratio_nf=ni / ci, ratio_delay=di / ci,
                         ratio_means_nf=np.nanmean(np.concatenate(list(nf.values()))) / cm,
                         ratio_means_delay=np.nanmean(np.concatenate(list(dl.values()))) / cm))
        if rd == "matched":
            print(col, "per-rat ranksums:"); print(t.round(5).to_string(index=False))
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))
print("control per-rat medians (b):", {r: round(np.nanmedian(ctrl[r].post_b), 2) for r in RATS})

**Result: partially reproduced, on three of the four rats.** Post-reward dwell is shorter on NF than on delay trials in gus, bernard and fievel (medians 3.88 vs 4.63, 4.61 vs 4.79, 2.76 vs 4.87 s; corrected p <= 0.002), and both mixed models give control trials longer (coefficients -6.26 and -5.09 s, p < 0.0001), under every plateau and leaving reading. The delay over control ratio of the inset means is 0.48 to 0.50, inside 0.4 to 0.6, but the NF over control ratio is 0.36 to 0.39 (ratio of means 0.37 to 0.40), just below the tolerance under every reading. Rat 1 (remy) cannot be tested, and its median would enter the inset mean.

## fig3a-speed-below-immobility

> Mean head speed while nosepoked before reward was below the 4 cm/s immobility cutoff for all trial types, and slightly but significantly lower on NF trials than on delay and control trials.

Per-trial mean speed over the pre-reward period (dfs_rips_vsctrlrats.m part 7), under each speed smoothing and plateau reading; the scale of 000629 is the assumed 0.2 cm per pixel.

In [ ]:
rows = []
for s, col in [(s, f"vel{s}") for s in nb.SPEED_SIGMAS] + [(s, f"vel{s}_last") for s in nb.SPEED_SIGMAS]:
    for rd in (READINGS if not col.endswith("_last") else ["matched"]):
        nf, dl = split(sel(rd), col)
        c = {r: ctrl[r][col].values for r in RATS}
        t = nb.ranksums_bh(nf, dl)
        l1, l2 = nb.lme_cohort(c, nf), nb.lme_cohort(c, dl)
        allv = list(nf.values()) + list(dl.values()) + list(c.values())
        rows.append(dict(sigma=s, start="last" if col.endswith("_last") else "first", plateau=rd, max_median=max(np.nanmedian(v) for v in allv), max_mean=max(np.nanmean(v) for v in allv), max_median_nf_cohort=max(np.nanmedian(v) for v in list(nf.values()) + list(dl.values())),
                         nf_slower_all=bool((t.median_nf < t.median_delay).all()), max_p_bh=t.p_bh.max(),
                         lme_nf_coef=l1[0], lme_nf_p=l1[1], lme_delay_coef=l2[0], lme_delay_p=l2[1]))
        if rd == "matched" and not col.endswith("_last"):
            print(f"sigma {s} s, per-rat medians NF and delay:", t[["rat", "median_nf", "median_delay", "p_bh"]].round(4).values.tolist())
print(pd.DataFrame(rows).round(5).to_string(index=False))

**Result: partially reproduced, on three of the four rats.** With the scale assumed for 000629 (0.2 cm per pixel), every rat and trial type is below 4 cm/s (largest median 3.02 cm/s, a control rat at 0.1 s smoothing; medians of the neurofeedback cohort's NF and delay trials at most 2.19 cm/s under every smoothing and plateau reading, 2.17 under the adopted one). NF trials are slower than delay trials in gus, bernard and fievel (corrected p <= 0.0004 under every reading), and NF trials are slower than control trials (mixed model -1.17 cm/s at 0.1 s, p < 1e-5; -0.56 cm/s at 0.5 s, p = 0.00002). The same holds with the pre-reward period started at the entry after the last withdrawal over 200 ms (largest median 3.02 cm/s; mixed model -1.17 cm/s at 0.1 s, p < 1e-5). At the assumed scale the x range of the head positions per run epoch has a smaller median in 000629 than in 000115 (112.6 against 115.7 cm) but a wider spread and a larger maximum (105.8 to 158.7 against 107.6 to 119.2 cm), so the scale is an assumption; the largest median of the neurofeedback cohort (2.19 cm/s) would stay below 4 cm/s under a scale 80% larger. The criterion holds in all three rats that can be tested; rat 1 (remy) cannot be.

## Behaviour per epoch (Figure 5)

Per epoch, among search trials (task phase <= 1) the fraction that are not errors (search efficiency), among repeat trials (phase > 1) the fraction that are not errors (repeat accuracy), and the fractions of redundant search trials and of repeat error trials that are NF trials (dfs_ripcontent_vsctrlrats.m part 5). The control epochs are all run epochs (gooddecode is not marked); the manipulation epochs are those of each plateau reading. Both readings of a redundant search (helper docstring) are run.

In [ ]:
def epoch_beh(d, col):
    rows = []
    for (r, day, ep), g in d.groupby(["rat", "day", "epoch"]):
        g = g[g[col].notna()]
        ph = g[col].values
        rep = ph > 1
        fr = np.mod(ph, 1)
        err = (fr > 0) & (fr < 0.85)
        row = dict(rat=r, day=day, epoch=ep)
        for typ, mask in (("R", g.port.values == "R"), ("W", g.port.values == "W"), ("all", np.ones(len(g), bool))):
            s_, rp = mask & ~rep, mask & rep
            row[f"search_eff_{typ}"] = (s_ & ~err).sum() / s_.sum() if s_.sum() else np.nan
            row[f"repeat_acc_{typ}"] = (rp & ~err).sum() / rp.sum() if rp.sum() else np.nan
        se, re_ = ~rep & err, rep & err
        row["redundant_frac_nf"] = ((g.port.values == "R") & se).sum() / se.sum() if se.sum() else np.nan
        row["repeat_err_frac_nf"] = ((g.port.values == "R") & re_).sum() / re_.sum() if re_.sum() else np.nan
        rows.append(row)
    return pd.DataFrame(rows)
fig5 = {}
for col in ("phase", "phase_b"):
    cb = epoch_beh(trials, col)
    for rd in READINGS:
        fig5[(col, rd)] = (cb, epoch_beh(sel(rd), col))
print("epochs per rat (matched plateau):", fig5[("phase", "matched")][1].groupby("rat").size().to_dict(),
      "; paper (Figure 4D, gooddecode): gus 14, bernard 18, fievel 26")

## fig5a-search-efficiency

> Manipulation subjects showed a slight improvement in search efficiency over control subjects, significant only on delay trials, with no difference between NF and delay trials.

In [ ]:
rows = []
for (col, rd), (cb, mb) in fig5.items():
    nf = {r: mb[mb.rat == r].search_eff_R.values for r in NFRATS}
    dl = {r: mb[mb.rat == r].search_eff_W.values for r in NFRATS}
    c = {r: cb[cb.rat == r].search_eff_all.values for r in RATS}
    t = nb.ranksums_bh(nf, dl)
    l1, l2 = nb.lme_cohort(c, nf), nb.lme_cohort(c, dl)
    rows.append(dict(redundant=col, plateau=rd, p_bh=t.p_bh.round(3).tolist(), lme_nf_coef=l1[0], lme_nf_p=l1[1],
                     lme_delay_coef=l2[0], lme_delay_p=l2[1], mean_ctrl=np.nanmean(np.concatenate(list(c.values()))),
                     mean_nf=np.nanmean(np.concatenate(list(nf.values()))), mean_delay=np.nanmean(np.concatenate(list(dl.values())))))
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced, resting on an assumption.** No NF against delay ranksum is significant (corrected p >= 0.25), and the NF against control model is not significant, as the claim says, but the delay against control model does not find the improvement: under the adopted readings the mean search efficiency is 0.732 for control, 0.719 for NF and 0.745 for delay epochs (coefficient +0.012, p = 0.65). Under reading (b) of a redundant search the delay coefficient is +0.048 to +0.050 with p = 0.062 to 0.14. No reading gives p < 0.05. Rat 1 (remy) is missing, and the control and plateau epochs are not the paper's gooddecode epochs.

## fig5b-redundant-search-balanced

> Redundant search trials were not biased toward either trial type; NF trials made up about half of them in each subject.

In [ ]:
rows = []
for (col, rd), (cb, mb) in fig5.items():
    for r in NFRATS:
        x = mb[mb.rat == r].redundant_frac_nf.values
        rows.append(dict(redundant=col, plateau=rd, rat=r, epochs=int(np.isfinite(x).sum()), median=np.nanmedian(x),
                         mean=np.nanmean(x), sign_p=nb.sign_test(x)))
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))
print("rats with p < 0.05:", res.groupby(["redundant", "plateau"]).apply(lambda d: int((d.sign_p < 0.05).sum()), include_groups=False).to_dict())

**Result: not reproduced, resting on an assumption.** Under the adopted plateau reading (matched), gus's redundant search trials lean to NF (median fraction 0.600, sign test p = 0.013; 0.004 under reading b), while bernard and fievel do not (p = 1.0 and 0.83). Under last7 and last10 no rat reaches p < 0.05 (smallest p = 0.26). The outcome rests on the epoch selection: the paper's test for rat 2 used 14 epochs, and 17 matched-plateau epochs have redundant search trials here. For gus the matched reading covers 5 days (26 to 30), below the 7 to 10 days of the Methods; it is adopted because its trial counts (688/643) match the paper's (684/640) within 1%, which neither the 7-day nor the 10-day reading does.

## fig5c-repeat-accuracy-unchanged

> On repeat trials the fraction of correct goal visits did not differ between the cohorts or between NF and delay trials.

In [ ]:
rows = []
for (col, rd), (cb, mb) in fig5.items():
    if col != "phase":
        continue  # repeat trials and their errors do not depend on the redundant-search reading
    nf = {r: mb[mb.rat == r].repeat_acc_R.values for r in NFRATS}
    dl = {r: mb[mb.rat == r].repeat_acc_W.values for r in NFRATS}
    c = {r: cb[cb.rat == r].repeat_acc_all.values for r in RATS}
    t = nb.ranksums_bh(nf, dl)
    l1, l2 = nb.lme_cohort(c, nf), nb.lme_cohort(c, dl)
    rows.append(dict(plateau=rd, p_bh=t.p_bh.round(3).tolist(), lme_nf_coef=l1[0], lme_nf_p=l1[1], lme_delay_coef=l2[0], lme_delay_p=l2[1],
                     mean_ctrl=np.nanmean(np.concatenate(list(c.values()))), mean_nf=np.nanmean(np.concatenate(list(nf.values()))),
                     mean_delay=np.nanmean(np.concatenate(list(dl.values())))))
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: partially reproduced, on three of the four rats.** No test finds a difference: corrected NF against delay ranksums p >= 0.43 and mixed models p >= 0.29 under every plateau reading (mean repeat accuracy 0.635 control, 0.668 NF, 0.710 delay under matched). Rat 1 (remy) cannot be tested.

## fig5d-repeat-errors-balanced

> Within the manipulation cohort, repeat error trials were equally distributed between NF and delay trials.

In [ ]:
rows = []
for (col, rd), (cb, mb) in fig5.items():
    if col != "phase":
        continue
    for r in NFRATS:
        x = mb[mb.rat == r].repeat_err_frac_nf.values
        rows.append(dict(plateau=rd, rat=r, epochs=int(np.isfinite(x).sum()), median=np.nanmedian(x), mean=np.nanmean(x), sign_p=nb.sign_test(x)))
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))
print("rats with p < 0.05:", res.groupby("plateau").apply(lambda d: int((d.sign_p < 0.05).sum()), include_groups=False).to_dict())

**Result: partially reproduced, on three of the four rats.** Under the adopted reading one rat, fievel (rat 4), has p < 0.05 (median fraction 0.556, p = 0.015), as in the paper (rat 4, p = 0.0227); gus and bernard do not (p = 0.80 and 0.81). Under last7 no rat and under last10 one rat (fievel) reaches p < 0.05. The pattern holds in the three rats that can be tested; rat 1 (remy) cannot be.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Claims on DANDI:000115 (9 claims: 5 tested, 4 blocked):

| Claim | Outcome |
|---|---|
| control-cohort-trial-counts | reproduced |
| control-decode-counts | partially reproduced |
| fig2s1h-control-dwell-increases | partially reproduced (assumption) |
| fig3a-control-speed | reproduced |
| fig5s1-control-search-efficiency-increases | reproduced (assumption) |
| 4 claims on the control cohort's SWRs and replay | blocked, too_large |

Claims whose data are elsewhere, tested on DANDI:000629 (26 claims: 7 tested, 19 blocked):

| Claim | Outcome |
|---|---|
| fig1s1c-prereward-dwell-matched | not reproduced (three rats, assumption) |
| fig2e-postreward-dwell | partially reproduced (three rats) |
| fig3a-speed-below-immobility | partially reproduced (three rats) |
| fig5a-search-efficiency | not reproduced (assumption) |
| fig5b-redundant-search-balanced | not reproduced (assumption) |
| fig5c-repeat-accuracy-unchanged | partially reproduced (three rats) |
| fig5d-repeat-errors-balanced | partially reproduced (three rats) |
| 19 claims on the neurofeedback cohort's SWRs and replay | blocked, too_large |

How the analysis differs from the paper's. The trials are parsed from the port DIO events by rules written here, since the Frank lab parser is not public; the trial counts match the paper's within 3% for the control rats, and for neurofeedback rats 2 to 4 a run of final whole days matches the paper's counts within 1%. Leaving the port, the start of the pre-reward period after a withdrawal, a redundant search trial, the position smoothing and the neurofeedback plateau are not defined in the paper or its code, and each is run under several readings. The mixed models are fit by maximum likelihood, as MATLAB fitlme does by default, with Wald p-values. The control epochs are all run epochs, as the files do not mark the paper's epoch flags. Rat 1 of the neurofeedback cohort could not be used. No claim on SWRs or replay was tested: the dandisets hold only the raw 30 kHz signal, and the detection or decoding would need about 1 to 17 TB of streaming per cohort (1.1 TB for one channel per tetrode in 000115, 1.9 TB in 000629, and 9.1 and 16.8 TB of whole files for decoding). The smallest subset that the Figure 2B criterion allows (two run epochs per control rat, 10 tetrodes each) is estimated at about 2 h of reading in four processes, over the notebook's limit of 60 min, and one run epoch per control rat at about 1 h, at the limit before detection is run.

What was found about the data. DANDI:000115 holds raw broadband, position, port events and statescript logs only; its electrode table gives no location, and the electrode groups say `ca1`, `bad_ca1`, `can1ref`, `lost` or nothing, but not which tetrodes are in the cell layer; the epoch flags the paper's code selects on are absent; position timestamps include NaN in 110 of 116 run epochs. In DANDI:000629 the port events of rat 1 (remy) from day 14 on, of its day 7, and of gus days 16 and 17 are corrupted (thousands of pump onsets per epoch, also in sleep epochs); the position conversion factor is 0; all of remy's sessions start on 1970-01-01; and nothing marks the neurofeedback phase, the plateau or the neurofeedback port (every run task is described as pretraining). The port is identified here from the yoked delays.